# Llama2 & Mistral AI efficient fine-tuning using QLoRA, bnb int4, gradient checkpointing and X—LLM 🦖

In this tutorial, we will:
- Fine-tune a 7B model using QLoRA (4bit) and Gradient checkpointing
- Save checkpoints (LoRA weights) to the Hugging Face Hub
- Fuse the LoRA weights into the main model
- Upload the resulting model (in int8) to the Hugging Face Hub

As a result, you will have a trained model that you can easily use in the following way:
```python
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("*MY_USERNAME*/*MY_COOL_MODEL*")
model = AutoModelForCausalLM.from_pretrained("*MY_USERNAME*/*MY_COOL_MODEL*")
```

LoRA parameter efficient finetuning method (be careful, this is a rather difficult method to understand, it's okay if you don't figure it out the first time):
- [blogpost](https://sebastianraschka.com/blog/2023/llm-finetuning-lora.html)
- [paper](https://arxiv.org/pdf/2106.09685.pdf)

Useful materials about `xllm`:
- [X—LLM Repo](https://github.com/BobaZooba/xllm): main repo of the `xllm` library
- [Quickstart](https://github.com/KompleteAI/xllm/tree/docs-v1#quickstart-): basics of `xllm`
- [Examples](https://github.com/BobaZooba/xllm/examples): minimal examples of using `xllm`
- [Guide](https://github.com/BobaZooba/xllm/blob/main/GUIDE.md): here, we go into detail about everything the library can
  do
- [Demo project](https://github.com/BobaZooba/xllm-demo): here's a minimal step-by-step example of how to use X—LLM and fit it
  into your own project
- [WeatherGPT](https://github.com/BobaZooba/wgpt): this repository features an example of how to utilize the xllm library. Included is a solution for a common type of assessment given to LLM engineers, who typically earn between $120,000 to $140,000 annually
- [Shurale](https://github.com/BobaZooba/shurale): project with a finetuned 7B Mistal model

# Installation

In [ ]:
!pip install xllm[deepspeed]

In [ ]:
!pip install deepspeed==0.13.1

# Login to HuggingFace to save model to the hub

You could find token here: https://huggingface.co/settings/tokens

YOUR_HF_TOKEN

In [ ]:
!huggingface-cli login

# [Optional] Login to W&B to save training process

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !wandb login

# Prepare

In [ ]:
import torch
import xllm

cuda_is_available = torch.cuda.is_available()

print(f"X—LLM version: {xllm.__version__}\nTorch version: {torch.__version__}\nCuda is available: {cuda_is_available}")
assert cuda_is_available

In [ ]:
from xllm import Config
from xllm.datasets import GeneralDataset
from xllm.experiments import Experiment

# Prepare dataset

In [ ]:
import csv

# File paths
file1_path = '/content/drive/MyDrive/NLP Thesis/Data/in-use/train-ctfl-cleaned.csv'
file2_path = '/content/drive/MyDrive/NLP Thesis/Data/generated_instruction.csv'
output_path = '/content/concatenated_file.csv'

# Read data from the first CSV file
data_from_file1 = []
with open(file1_path, 'r', newline='') as file1:
    reader = csv.reader(file1)
    for row in reader:
        data_from_file1.append(row)

# Read data from the second CSV file, skipping the first row
data_from_file2 = []
with open(file2_path, 'r', newline='') as file2:
    reader = csv.reader(file2)
    next(reader)  # Skip the first row
    for row in reader:
        data_from_file2.append(row)

# Concatenate data from both files
concatenated_data = data_from_file1 + data_from_file2

# Write the concatenated data to a new CSV file
with open(output_path, 'w', newline='') as outfile:
    writer = csv.writer(outfile)
    writer.writerows(concatenated_data)

print("CSV files concatenated successfully.")


TRAIN

In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files= "/content/concatenated_file.csv")

In [ ]:
dataset = dataset["train"]

In [ ]:
dataset

In [ ]:
import pandas as pd
def format_data(data):
  df = pd.DataFrame(data)
  defaultPrompt = '''แก้ไขประโยคต่อไปนี้ให้มีการใช้ภาษาที่ถูกต้อง โดยแก้ไขเฉพาะที่จำเป็น'''
  df['text'] =  '### Instruction: ' + defaultPrompt + '\n\n### Input: ' + df['text'] + '\n\n### Output: ' + df['expected']
  return df['text']

In [ ]:
testdata = format_data(dataset)
testdata

In [ ]:
train_data = list()

for sample in testdata:
    train_data.append({"text": sample})

In [ ]:
len(train_data)

In [ ]:
train_data[10:13]

In [ ]:
train_dataset = GeneralDataset(data=train_data, separator="\n\n")

In [ ]:
train_dataset[11]

EVAL

In [ ]:
dataset = load_dataset("csv", data_files= "/content/drive/MyDrive/NLP Thesis/Data/in-use/dev-ctfl-cleaned.csv")
dataset = dataset["train"]

In [ ]:
dataset

In [ ]:
testdata = format_data(dataset)
testdata

In [ ]:
train_data = list()

for sample in testdata:
    train_data.append({"text": sample})

In [ ]:
len(train_data)

In [ ]:
train_data[10:13]

In [ ]:
eval_dataset = GeneralDataset(data=train_data, separator="\n\n")

In [ ]:
eval_dataset[11]

# Make some constants

- `backbone_model_name` - the name of the model from Hugging Face Hub that you want to finetune. For example Mistral or Llama 2
- `push_to_hub_while_training` - set to True to save every model checkpoint. For this, you will definitely need to log in to the Hugging Face Hub using the command: `!huggingface-cli login`
- `lora_hub_model_id` - the name of your future repository in Hugging Face Hub where the LoRA weights will be saved. Format: *USERNAME*/*REPO_NAME*
- `hub_model_id` - the name of your future repository in Hugging Face Hub where you will save the final fused model. Format: *USERNAME*/*REPO_NAME*
- `max_steps` - the maximum number of training steps that your model will go through
- `save_steps` - frequency of checkpoint saving.
- `warmup_steps` - the number of training steps for which the model will warm up. Usually, this is 5-10% of max_steps
- `report_to_wandb` - set to True in order to track the training process of the model using W&B. For this, you will definitely need to log in to W&B using the command: `!wandb login`
- `wandb_project` - project name at W&B
- `wandb_entity` - your username or your company username at W&B


The maximum tested model size for this tutorial in Colab is 7B.

### Note about sharded

Large models in Hugging Face Hub are typically split into several files - shards. This is done so as not to load the entire model at once, but to load it piece by piece. Models are usually divided into shards of 10 gigabytes each, but unfortunately, that's not suitable for Colab. This is because the free Colab notebooks have too little RAM, and the notebook crashes during model loading.

Therefore, use models whose shards take up less memory, for example, 3 gigabytes or less. This way, the model loading will proceed without errors due to the peculiarities of quantization. Usually, you can find almost any popular model with small-sized shards that are suitable for Colab. Just look in the Hugging Face Hub in the format MODEL_NAME-sharded.

For example:
mistralai/Mistral-7B-v0.1 -> bn22/Mistral-7B-v0.1-sharded meta-llama
Llama-2-7b-hf -> TinyPixel/Llama-2-7B-bf16-sharded

In [ ]:
# # model must be sharded
# backbone_model_name = "TinyPixel/Llama-2-7B-bf16-sharded"
backbone_model_name = "scb10x/typhoon-7b"#"openthaigpt/openthaigpt-1.0.0-beta-13b-chat-hf" #"scb10x/typhoon-7b"

push_to_hub_while_training = True
lora_hub_model_id = "pakawadeep/typhoon-7b-lora-ctfl-th-gec-100st-augmented"
hub_model_id = "pakawadeep/typhoon-7b-ctfl-th-gec-100st-augmented"

max_steps = 100
save_steps = 25
warmup_steps = 10

report_to_wandb = False
wandb_project = None
wandb_entity = None

In [ ]:
if report_to_wandb and wandb_project:
    print("Please set at least wandb_project for W&B tracking. wandb_entity is your or your company username at W&B")

# Make a X—LLM config

More information about config you could find here: https://github.com/BobaZooba/xllm/blob/main/GUIDE.md#config

> Important note! When using DeepSpeed, checkpoints will currently take significantly longer to save with LoRA because DeepSpeed also saves the entire model. This behavior is expected and is planned to be addressed in future versions.

In [ ]:
config = Config(
    use_gradient_checkpointing=True,
    model_name_or_path=backbone_model_name,
    use_flash_attention_2=False,  # not supported in colab
    load_in_4bit=True,
    prepare_model_for_kbit_training=True,
    apply_lora=True,
    warmup_steps=warmup_steps,
    max_steps=max_steps + 1,
    save_steps=save_steps,
    logging_steps=1,
    num_train_epochs=1,

    #do_eval=True,
    #eval_steps=25,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    max_length=2048,

    # DeepSpeed reduce memory requirements by offloading
    deepspeed_stage="stage_2",
    single_gpu=True,

    # tokenizer_padding_side="right",  # good for llama2

    push_to_hub=push_to_hub_while_training,
    hub_model_id=lora_hub_model_id,
    hub_private_repo=False,

    # W&B
    report_to_wandb=False,
    wandb_project=wandb_project,
    wandb_entity=wandb_entity,
)

# Make a X—LLM experiment

In [ ]:
experiment = Experiment(config=config, train_dataset=train_dataset)#, eval_dataset=eval_dataset)

## Build experiment

In [ ]:
# Build Experiment from Config: init tokenizer and model, apply LoRA and so on
experiment.build()

In [ ]:
# Run training
experiment.run()

## If you have not pushed checkpoints to Hugging Face Hub, now you need to push the last checkpoint there

To do this, you will definitely need to log in to the Hugging Face Hub

In [ ]:
experiment.push_to_hub(repo_id="pakawadeep/typhoon-7b-ctfl-th-gec-100st-augmented")

In [ ]:
#!huggingface-cli login

In [ ]:
# experiment.push_to_hub(
#     repo_id=lora_hub_model_id,
#     private=False,
# )

# Fuse LoRA weight and push fused model to the Hugging Face Hub

For fusing, we will be using bitsandbytes int8, using this line in the config:

```python
load_in_8bit=True
```

This will reduce memory consumption and slightly speed up the model, but it will marginally deteriorate its quality. In the free version of Colab, only this option is available due to RAM limitations. You can perform the same operation with `fp16` by removing the line above from the config and running the fusing on a machine with more RAM.

### Why can't we fuse the model we just trained?

We have access to the model:

```python
experiment.model
```

But we loaded the given model in int4, and currently, saving the model in int4 is not available. Formally, we can fuse an int4 model, but we will not be able to save it either locally or on Hugging Face Hub. Therefore, we will reload the model and the trained LoRA adapter afresh and fuse it into int8.

## Free up memory for fusing

There is not much memory in the Colab. It will crash if it is not done.

> ⚠️ Important note!
If you've used DeepSpeed, it won't work because the memory isn't freed up in the necessary amount due to DeepSpeed. So if you're using DeepSpeed, please do a "Restart session" and continue from there.

In [ ]:
import gc
import torch

In [ ]:
del experiment

gc.collect()
torch.cuda.empty_cache()

## Fusing

In [ ]:
from xllm import fuse, Config

In [ ]:
# backbone_model_name = "alexsherstinsky/Mistral-7B-v0.1-sharded"
# lora_hub_model_id = "BobaZooba/AntModel-7B-XLLM-Demo-LoRA"
# hub_model_id = "BobaZooba/AntModel-7B-XLLM-Demo"

In [ ]:
fusing_config = Config(
    model_name_or_path=backbone_model_name,
    lora_hub_model_id=lora_hub_model_id,
    load_in_8bit=True,
    push_to_hub=True,
    hub_model_id=hub_model_id,
    hub_private_repo=False,
    max_shard_size="1GB", # to use fused model later at colab
)

In [ ]:
tokenizer, model = fuse(fusing_config)

## Generate from model

In [ ]:
input_text = """### Instruction: แก้ไขประโยคต่อไปนี้ให้มีการใช้ภาษาที่ถูกต้อง โดยแก้ไขเฉพาะที่จำเป็น
### Input: งานแต่งงานของเขาจะจัดขึ้นในหน้าเดือนหน้า
### Output:"""

In [ ]:
tokenized = tokenizer(input_text, return_tensors="pt").to("cuda:0")

In [ ]:
generated_indices = model.generate(
    **tokenized,
    max_new_tokens=64,
    do_sample=True,
    temperature=0.1,
).cpu()

In [ ]:
tokenizer.batch_decode(generated_indices)

In [ ]:
print(f"You could find your model at: https://huggingface.co/{hub_model_id}")

# 🎉 You are awesome!

## Now you know how to prototype models using `xllm`

### Explore more examples at X—LLM repo

https://github.com/BobaZooba/xllm

Useful materials:

- [X—LLM Repo](https://github.com/BobaZooba/xllm): main repo of the `xllm` library
- [Quickstart](https://github.com/KompleteAI/xllm/tree/docs-v1#quickstart-): basics of `xllm`
- [Examples](https://github.com/BobaZooba/xllm/examples): minimal examples of using `xllm`
- [Guide](https://github.com/BobaZooba/xllm/blob/main/GUIDE.md): here, we go into detail about everything the library can
  do
- [Demo project](https://github.com/BobaZooba/xllm-demo): here's a minimal step-by-step example of how to use X—LLM and fit it
  into your own project
- [WeatherGPT](https://github.com/BobaZooba/wgpt): this repository features an example of how to utilize the xllm library. Included is a solution for a common type of assessment given to LLM engineers, who typically earn between $120,000 to $140,000 annually
- [Shurale](https://github.com/BobaZooba/shurale): project with the finetuned 7B Mistal model



In [ ]:
import pandas as pd

# Assuming you have already initialized your tokenizer and model
data = []
# Read the CSV file
df = pd.read_csv("/content/drive/MyDrive/NLP Thesis/Data/ctfl_cleaned_test.csv")

# Iterate over rows
for index, row in df.iterrows():
    # Get text from column 2
  text_from_column_2 = row['text']

    # Generate the desired text format
  input_text = f"### Instruction: แก้ไขประโยคต่อไปนี้ให้มีการใช้ภาษาที่ถูกต้อง โดยแก้ไขเฉพาะที่จำเป็น\n### Input: {text_from_column_2}\n### Output: "

    # Tokenize the input text
  tokenized = tokenizer(input_text, return_tensors="pt").to("cuda:0")
  generated_indices = model.generate(
      **tokenized,
      max_new_tokens=64,
      do_sample=True,
  ).cpu()

  result = tokenizer.batch_decode(generated_indices)
  print(result)

  data.append([text_from_column_2, result])



In [ ]:
import csv

In [ ]:
filename = '/content/drive/MyDrive/NLP Thesis/Data/typhoon_7B_100steps_test_augmented.csv'

# Write the data to a CSV file
with open(filename, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # Write the header row
    writer.writerow(['sentence', 'predicted'])
    # Write the data rows
    writer.writerows(data)

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("/content/drive/MyDrive/NLP Thesis/Data/typhoon_7B_100steps_test_augmented.csv")

# Iterate over rows
for index, row in df.iterrows():
    # Get text from column 2
    text_from_column_2 = row['predicted']

    # Find the position of "### Output:"
    output_index = text_from_column_2.find("### Output:")
    if output_index != -1:
        # Extract the text after "### Output:"
        output_text = text_from_column_2[output_index + len("### Output:"):]

        # Remove the surrounding '<s>' and '</s>' tags
        output_text = output_text.strip().strip('<s>').strip('</s>]').strip()
        output_text = output_text.strip("</s>'")

        # Update the original column with the modified output
        df.at[index, 'predicted'] = output_text

# Save the modified DataFrame back to the CSV file
df.to_csv("/content/drive/MyDrive/NLP Thesis/Evaluation/typhoon_7B_100steps_test_augmented.csv", index=False)
